<a href="https://colab.research.google.com/github/hawey2/Day21-Track3-Finetuning-Lab-NguyenNgocHan-2A202602511/blob/main/colab/Lab21_01_data_and_mask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Setup (chạy ô này trước)
# Colab bắt đầu với một máy trống — clone repo và cài dependency.
import os, subprocess, sys

REPO = "https://github.com/VinUni-AI20k/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("Day21-Track3-Finetuning-Lab")
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

os.environ.setdefault("COMPUTE_TIER", "T4")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime > Change runtime type > T4 GPU")


GPU: NONE — Runtime > Change runtime type > T4 GPU


# NB1 — Dữ liệu, chat template & mask

**Chạy được trên CPU. Không cần GPU.** Đây là notebook duy nhất như vậy — và cũng là
notebook quyết định kết quả của cả lab.

> Deck §17.2: *che loss và chat template quyết định kết quả nhiều hơn mọi biến thể
> LoRA cộng lại.* Notebook này không dạy bạn tin điều đó — nó bắt bạn **nhìn thấy** nó.

Cuối notebook bạn sẽ có 4 artefact bắt buộc nộp:
1. `results/mask_proof.json` — bằng chứng mask đúng
2. `results/template_check.json` — template có nuốt khối `<think>` không
3. `results/token_stats.json` — p95 → `max_length`
4. `data/split/{train,val}.jsonl` — split cố định seed=42

In [2]:
import json
import os
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
sys.path.insert(0, str(pathlib.Path.cwd() / "src"))

from labkit import data, report
from labkit.config import get_tier

ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / "data").exists() else pathlib.Path.cwd().parent
TIER = get_tier(os.environ.get("COMPUTE_TIER", "T4"))
print(f"tier={TIER.name}  model={TIER.model_id}  max_length={TIER.max_length}")

tier=T4  model=unsloth/Qwen3.5-4B  max_length=1024


## 1. Nạp corpus

Corpus đi kèm lab: ticket CSKH tiếng Việt → JSON triage 4 trường. Bạn **được khuyến
khích** thay bằng dữ liệu miền của mình (xem §"Đổi dataset" trong README) — nhưng hãy
chạy hết notebook này một lượt với corpus mặc định trước, để có mốc so sánh.

In [3]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


train_raw = load_jsonl(ROOT / "data" / "train_seed.jsonl")
print(f"{len(train_raw)} mẫu huấn luyện")
print(json.dumps(train_raw[0], ensure_ascii=False, indent=2)[:400])

250 mẫu huấn luyện
{
  "instruction": "Phân loại ticket chăm sóc khách hàng sau thành JSON với đúng 4 khóa: intent, urgency, product, sentiment. Chỉ trả về JSON, không giải thích.\n\nintent thuộc: doi_tra | van_chuyen | hoan_tien | san_pham_loi | hoi_thong_tin\nurgency thuộc: cao | trung_binh | thap\nsentiment thuộc: tieu_cuc | trung_tinh | tich_cuc\nproduct: tên sản phẩm xuất hiện trong ticket.",
  "input": "Alo sh


## 2. Tokenizer + kiểm tra template

Chỉ tải **file tokenizer** (vài MB), không tải trọng số. Chạy được trên máy không GPU.

In [4]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(TIER.model_id, trust_remote_code=True)
print("eos_token:", tok.eos_token)

config.json:   0%|          | 0.00/2.76k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/15.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 20.0MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.99k [00:00<?, ?B/s]

eos_token: <|im_end|>


### Kiểm tra bắt buộc #1 — template có giữ khối suy luận không?

Deck §22: **một số chat template xoá nội dung `<think>` ngay trong
`apply_chat_template`.** Khi đó reasoning traces trong dataset của bạn *không bao giờ*
tới được hàm loss — và không có gì báo lỗi cả. Kiểm tra một lần cho mỗi base model.

In [5]:
check = data.thinking_survives(tok)
print("VERDICT:", check["verdict"])
print("\n--- chuỗi đã render ---")
print(check["rendered"])
report.write_json(check, "template_check.json", results_dir=ROOT / "results")

VERDICT: reasoning preserved — safe to train on traces

--- chuỗi đã render ---
<|im_start|>user
2+2?<|im_end|>
<|im_start|>assistant
<think>
buoc 1: kiem tra. buoc 2: tra loi.
</think>

4<|im_end|>



PosixPath('/content/Day21-Track3-Finetuning-Lab/results/template_check.json')

## 3. Xây mask — và ĐỌC nó

Bốn chế độ, và bạn phải hiểu khác biệt trước khi train:

| mode | Loss tính trên | Dùng khi |
|---|---|---|
| `assistant-only` | toàn bộ lượt assistant | mặc định SFT |
| `masked-think` | lượt assistant **trừ** khối suy luận | base có chế độ thinking (§17.5) |
| `response-only` | chỉ phần sau `</think>` | nghiêm ngặt nhất |
| `everything` | **cả prompt** | ✗ đây là bug kinh điển — để bạn nhìn thấy nó |

In [6]:
sample = data.to_messages(train_raw[0])

for mode in ("assistant-only", "everything"):
    ex = data.build_example(tok, sample, max_length=TIER.max_length, mask_mode=mode)
    print("=" * 70)
    print(f"mode = {mode}   supervised {ex.n_supervised}/{ex.n_total} "
          f"({ex.supervised_fraction:.0%})")
    print("--- LOSS TÍNH TRÊN ĐOẠN NÀY ---")
    print(data.decode_supervised(tok, ex)[:400])

mode = assistant-only   supervised 39/94 (41%)
--- LOSS TÍNH TRÊN ĐOẠN NÀY ---
</think>

{"intent": "doi_tra", "urgency": "trung_binh", "product": "balo laptop", "sentiment": "trung_tinh"}<|im_end|>

mode = everything   supervised 94/94 (100%)
--- LOSS TÍNH TRÊN ĐOẠN NÀY ---
<|im_start|>system
Phân loại ticket sau.<|im_end|>
<|im_start|>user
Alo shop, mình đặt balo laptop mã đơn VN411453. Cho tôi trả lại. Đã 3 ngày rồi. Cho tôi hỏi.<|im_end|>
<|im_start|>assistant
<think>

</think>

{"intent": "doi_tra", "urgency": "trung_binh", "product": "balo laptop", "sentiment": "trung_tinh"}<|im_end|>



**Dừng lại và đọc kỹ output ở trên.**

Với `everything`, câu hỏi của bạn nằm trong phần được tính loss → model sẽ học cách
*viết lại câu hỏi*. Đó chính xác là triệu chứng ở deck §22 (“Model viết tiếp câu hỏi
của bạn”). Rất nhiều người chỉ phát hiện ra sau khi train xong 3 tiếng.

### Kiểm tra bắt buộc #2 — mask proof

Đây là artefact nộp bài. Nó khẳng định: phần được tính loss **chứa** câu trả lời và
**không chứa** câu hỏi.

In [7]:
ex = data.build_example(tok, sample, max_length=TIER.max_length, mask_mode="assistant-only")
supervised = data.decode_supervised(tok, ex)
masked = data.decode_masked(tok, ex)

answer = sample[-1]["content"][:40]
question_fragment = train_raw[0]["input"][:40]

proof = {
    "mask_mode": "assistant-only",
    "n_supervised": ex.n_supervised,
    "n_total": ex.n_total,
    "supervised_fraction": round(ex.supervised_fraction, 4),
    "answer_is_supervised": answer in supervised,
    "question_is_masked": question_fragment not in supervised,
    "supervised_preview": supervised[:300],
    "masked_preview": masked[:300],
}
assert proof["answer_is_supervised"], "câu trả lời KHÔNG nằm trong loss — mask sai"
assert proof["question_is_masked"], "câu hỏi ĐANG nằm trong loss — mask sai"
print(json.dumps({k: v for k, v in proof.items() if not k.endswith("preview")},
                 ensure_ascii=False, indent=2))
report.write_json(proof, "mask_proof.json", results_dir=ROOT / "results")

{
  "mask_mode": "assistant-only",
  "n_supervised": 39,
  "n_total": 94,
  "supervised_fraction": 0.4149,
  "answer_is_supervised": true,
  "question_is_masked": true
}


PosixPath('/content/Day21-Track3-Finetuning-Lab/results/mask_proof.json')

## 4. Độ dài token → `max_length`

Deck §17: `max_length` là **số đo**, không phải con số đoán. Đặt theo p95 rồi làm tròn
lên luỹ thừa 2. Đặt quá lớn = trả tiền cho padding; quá nhỏ = cắt mất câu trả lời.

In [8]:
lengths = [
    data.build_example(tok, data.to_messages(r), max_length=8192).n_total
    for r in train_raw
]
stats = data.token_stats(lengths)
print(json.dumps(stats, ensure_ascii=False, indent=2))
report.write_json(stats, "token_stats.json", results_dir=ROOT / "results")

if stats["suggested_max_length"] != TIER.max_length:
    print(f"\n⚠ p95 gợi ý max_length={stats['suggested_max_length']} "
          f"nhưng tier đang đặt {TIER.max_length}. Ghi lại lựa chọn của bạn trong REPORT.md.")

{
  "n": 250,
  "mean": 93.1,
  "p50": 93,
  "p95": 98,
  "p99": 100,
  "max": 101,
  "suggested_max_length": 256
}

⚠ p95 gợi ý max_length=256 nhưng tier đang đặt 1024. Ghi lại lựa chọn của bạn trong REPORT.md.


## 5. Split cố định

`seed=42` ở mọi notebook. Hai lần chạy khác seed thì **không so sánh được với nhau** —
và cả lab này là về việc so sánh.

In [9]:
train, val = data.split(train_raw, train_frac=0.9, seed=42)
split_dir = ROOT / "data" / "split"
split_dir.mkdir(exist_ok=True)
for name, rows in (("train", train), ("val", val)):
    with (split_dir / f"{name}.jsonl").open("w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"train={len(train)}  val={len(val)}  -> {split_dir}")

train=225  val=25  -> /content/Day21-Track3-Finetuning-Lab/data/split


## ✅ Checkpoint NB1

- [ ] `results/template_check.json` — biết template có giữ `<think>` không
- [ ] `results/mask_proof.json` — hai assert đều xanh
- [ ] `results/token_stats.json` — có p95
- [ ] `data/split/{train,val}.jsonl` — seed 42

→ Sang **NB2**: đóng băng tập eval và đo ba baseline **trước khi** train.